<a href="https://colab.research.google.com/github/cfan27/PY191_lab/blob/main/Copy_of_July_15_final_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression: Predicting College Admission

## Goal
In this activity, you will use **logistic regression** to predict whether a student is admitted to a college based on two exam scores.

You will build the model yourself, then use scikit-learn to check whether your implementation reaches similar results.

By the end, you should be able to:

1. Explain why logistic regression is used for a **yes/no** prediction.
2. Explain how gradient descent trains a model.
3. Build and train a simple logistic regression model.
4. Interpret a predicted probability.
5. Measure the model's accuracy.
6. Describe a decision boundary.
7. Compare a hand-built model with a reference implementation.

## What you need to know
You only need basic Python skills:

- running notebook cells
- reading variables
- changing a small amount of code
- interpreting graphs and numbers

All required data are created inside this notebook. No files need to be downloaded.


## 1. Setup

Run the cell below. It imports the tools used in this notebook.

- **NumPy** creates and stores numerical data.
- **Matplotlib** makes graphs.
- **scikit-learn** is used only at the end to check our work.

The main logistic regression model will be built using NumPy.


In [1]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Make random results repeatable.
np.random.seed(7)

print("Setup complete!")


Setup complete!


## 2. Create the data

Each student has:

- an Exam 1 score
- an Exam 2 score
- an admission result: `0` means not admitted and `1` means admitted

The code below creates a small, realistic-looking dataset. The admission results are not perfectly predictable, just like real data.

In [ ]:
number_of_students = 120

exam_1 = np.random.randint(45, 101, number_of_students)
exam_2 = np.random.randint(45, 101, number_of_students)

# Higher combined scores usually mean a better chance of admission.
combined_score = exam_1 + exam_2
random_noise = np.random.normal(0, 10, number_of_students)
admitted = (combined_score + random_noise >= 145).astype(int)

# X contains the two input features. y contains the yes/no result.
X = np.column_stack((exam_1, exam_2))
y = admitted

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("First five students:")
print(X[:5])
print("First five results:", y[:5])

### Question 1

What does one row of `X` represent? What does one value of `y` represent?

**Your answer:**

_Write your answer here._

## 3. Visualize the data

A scatter plot helps us see whether admitted and non-admitted students tend to have different exam scores.

In [ ]:
not_admitted = y == 0
is_admitted = y == 1

plt.figure(figsize=(8, 6))
plt.scatter(X[not_admitted, 0], X[not_admitted, 1],
            label="Not admitted", marker="o")
plt.scatter(X[is_admitted, 0], X[is_admitted, 1],
            label="Admitted", marker="+")

plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.title("College Admission Data")
plt.legend()
plt.grid(True)
plt.show()

### Question 2

Describe the pattern in the graph. Do students with higher scores appear more likely to be admitted?

**Your answer:**

_Write your answer here._

## 4. Why logistic regression?

The result has only two possible classes:

- `0`: not admitted
- `1`: admitted

Logistic regression first calculates a score, then changes that score into a probability using the **sigmoid function**:

$$
\text{sigmoid}(z) = \frac{1}{1 + e^{-z}}
$$

The sigmoid always gives a number between 0 and 1, so it can be interpreted as a probability.

In [ ]:
def sigmoid(z):
    # TODO: implement the sigmoid function
    return 1 / (1 + np.exp(-z))

z_values = np.linspace(-8, 8, 200)
probabilities = sigmoid(z_values)

plt.figure(figsize=(8, 5))
plt.plot(z_values, probabilities)
plt.axhline(0.5, linestyle="--")
plt.xlabel("Model score, z")
plt.ylabel("Probability")
plt.title("The Sigmoid Function")
plt.grid(True)
plt.show()

print("sigmoid(-5) =", round(sigmoid(-5), 3))
print("sigmoid(0)  =", round(sigmoid(0), 3))
print("sigmoid(5)  =", round(sigmoid(5), 3))

### Question 3

What probability does the sigmoid function produce when `z = 0`?

**Your answer:**

_Write your answer here._

## 5. Split the data

We will use:

- **training data** to teach the model
- **test data** to check the model on students it has not seen before

Here, 75% of the students are used for training and 25% are used for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=7,
    stratify=y
)

print("Training students:", len(X_train))
print("Testing students:", len(X_test))

### Question 4

Why should we test the model on students that were not used to train it?

**Your answer:**

_Write your answer here._

## 6. Build logistic regression ourselves

A logistic regression model learns three numbers:

- one weight for Exam 1
- one weight for Exam 2
- one bias

For each student, it first calculates a score:

$$
z = w_1x_1 + w_2x_2 + b
$$

It then sends that score through the sigmoid function to get a probability.

At the beginning, the weights and bias are all zero. The model improves them a little at a time.


### Scale the exam scores

The exam scores are much larger than the weights we want to learn. Scaling makes training easier and more stable.

We calculate the mean and standard deviation from the **training data only**, then use those same values for both the training and test data.


In [ ]:
# TODO: calculate the training mean and standard deviation
feature_mean = X_train.mean(axis=0)
feature_std = X_train.std(axis=0)

# TODO: scale the training and test data
X_train_scaled = (X_train - feature_mean) / feature_std
X_test_scaled = (X_test - feature_mean) / feature_std

print("Mean of scaled training data:", np.round(X_train_scaled.mean(axis=0), 3))
print("Standard deviation of scaled training data:", np.round(X_train_scaled.std(axis=0), 3))

### How training works

The model repeats these four steps:

1. Make a probability prediction for every training student.
2. Compare each prediction with the correct answer.
3. Calculate how the weights and bias should change.
4. Update the weights and bias.

This repeated adjustment process is called **gradient descent**.

The model's error is measured with **log loss**. A smaller loss means the predicted probabilities are closer to the correct answers.


## Gradient descent

For logistic regression,

$$
p=\sigma(Xw+b)
$$

and the binary cross-entropy loss is

$$
L=-\frac{1}{n}\sum_{i=1}^{n}
\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right].
$$

The gradients are

$$
\frac{\partial L}{\partial w}=\frac{1}{n}X^T(p-y)
$$

and

$$
\frac{\partial L}{\partial b}=\frac{1}{n}\sum_{i=1}^{n}(p_i-y_i).
$$

Update the parameters with

$$
w\leftarrow w-\alpha\frac{\partial L}{\partial w},
\qquad
b\leftarrow b-\alpha\frac{\partial L}{\partial b}.
$$

Use these formulas directly in the training loop.


In [ ]:
class MyLogisticRegression:
    def __init__(self, learning_rate=0.1, number_of_steps=1000):
        self.learning_rate = learning_rate
        self.number_of_steps = number_of_steps

    def fit(self, X, y):
        number_of_students, number_of_features = X.shape

        # TODO: initialize the weights and bias
        self.weights = np.zeros(number_of_features)
        self.bias = 0
        self.loss_history = []

        for step in range(self.number_of_steps):
            # 1. Compute scores and probabilities
            scores = np.dot(X, self.weights) + self.bias
            probabilities = 1 / (1 + np.exp(-scores))

            # 2. Compute binary cross-entropy loss
            tiny_number = 1e-9
            loss = -np.mean(y * np.log(probabilities + tiny_number) + (1 - y) * np.log(1 - probabilities + tiny_number))
            self.loss_history.append(loss)

            # 3. Compute gradients
            delta = probabilities - y
            weight_gradient = np.dot(X.T, delta) / number_of_students
            bias_gradient = np.sum(delta) / number_of_students

            # 4. Update parameters
            self.weights -= self.learning_rate * weight_gradient
            self.bias -= self.learning_rate * bias_gradient

        return self

    def predict_proba(self, X):
        scores = np.dot(X, self.weights) + self.bias
        admission_probability = 1 / (1 + np.exp(-scores))
        return np.column_stack((1 - admission_probability, admission_probability))

    def predict(self, X):
        admission_probability = self.predict_proba(X)[:, 1]
        return (admission_probability >= 0.5).astype(int)

### Question 5

Find the four numbered training steps inside `fit()`.

Which two lines actually update the weights and bias?

**Your answer:**

_Write your answer here._


## 7. Train the model

The `learning_rate` controls how large each update is.

The `number_of_steps` controls how many times the model updates its weights and bias.


In [ ]:
model = MyLogisticRegression(
    learning_rate=0.1,
    number_of_steps=1000
)

# TODO: train the model
model.fit(X_train_scaled, y_train)

print("The model has been trained.")
print("Starting loss:", round(model.loss_history[0], 4))
print("Final loss:", round(model.loss_history[-1], 4))
print("Learned weights:", np.round(model.weights, 3))
print("Learned bias:", round(model.bias, 3))

### Plot the training loss

The loss should generally decrease as the model learns.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(model.loss_history)
plt.xlabel("Training step")
plt.ylabel("Average log loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()


### Question 6

What happens to the loss during training? What does this tell you about the model?

**Your answer:**

_Write your answer here._


## 8. Make predictions

`predict()` gives a final class, either `0` or `1`.

`predict_proba()` gives the probability of each class.


In [ ]:
# TODO: make predictions on the test set
predicted_classes = model.predict(X_test_scaled)
predicted_probabilities = model.predict_proba(X_test_scaled)

print("First five test students:")
print(X_test[:5])
print()
print("Predicted classes:", predicted_classes[:5])
print("Probability of admission:", np.round(predicted_probabilities[:5, 1], 3))
print("Actual results:", y_test[:5])

### Question 7

For the first test student, compare the predicted probability, predicted class, and actual result. Was the model correct?

**Your answer:**

_Write your answer here._


## 9. Evaluate the model

Accuracy is the fraction of predictions that are correct.

A confusion matrix counts four types of results:

- true negatives: correctly predicted not admitted
- false positives: predicted admitted, but actually not admitted
- false negatives: predicted not admitted, but actually admitted
- true positives: correctly predicted admitted


In [ ]:
# TODO: calculate accuracy and confusion matrix
accuracy = accuracy_score(y_test, predicted_classes)
matrix = confusion_matrix(y_test, predicted_classes)

print("Accuracy:", round(accuracy, 3))
print("Accuracy as a percent:", round(accuracy * 100, 1), "%")
print()
print("Confusion matrix:")
print(matrix)

### Question 8

1. What percent of the test predictions were correct?
2. How many test students were classified incorrectly?

Hint: the incorrect predictions are the two off-diagonal values in the confusion matrix.

**Your answer:**

_Write your answer here._


## 10. Check our work with scikit-learn

Now we will train scikit-learn's logistic regression model using the exact same training and test data.

This is a **reference check**, not the main exercise. The results do not have to match perfectly because the two models may use different optimization methods and stopping rules.


In [ ]:
reference_model = LogisticRegression(
    penalty=None,
    max_iter=10000
)
reference_model.fit(X_train_scaled, y_train)

reference_classes = reference_model.predict(X_test_scaled)
reference_probabilities = reference_model.predict_proba(X_test_scaled)

# TODO: compare your model with scikit-learn
reference_accuracy = accuracy_score(y_test, reference_classes)
prediction_agreement = np.mean(predicted_classes == reference_classes)
average_probability_difference = np.mean(np.abs(predicted_probabilities[:, 1] - reference_probabilities[:, 1]))

print("Our model accuracy:", round(accuracy, 3))
print("scikit-learn accuracy:", round(reference_accuracy, 3))
print("Prediction agreement:", round(prediction_agreement * 100, 1), "%")
print("Average probability difference:", round(average_probability_difference, 4))

### Question 9

Are the two accuracies similar? On what percent of test students do the models agree?

Would you consider the hand-built implementation successful? Explain.

**Your answer:**

_Write your answer here._


## 11. Try your own student

Change the two exam scores below, then run the cell.

New students must be scaled using the same mean and standard deviation that were calculated from the training data.


In [ ]:
student_exam_1 = 78
student_exam_2 = 82

new_student = np.array([[student_exam_1, student_exam_2]])
new_student_scaled = (
    new_student - feature_mean
) / feature_std

admission_probability = model.predict_proba(
    new_student_scaled
)[0, 1]
predicted_result = model.predict(
    new_student_scaled
)[0]

print("Predicted probability of admission:",
      round(admission_probability * 100, 1), "%")
print("Predicted class:", predicted_result)

if predicted_result == 1:
    print("Prediction: admitted")
else:
    print("Prediction: not admitted")


### Question 10

Try at least three pairs of exam scores. Record the scores and predicted probabilities below.

| Exam 1 | Exam 2 | Predicted probability of admission |
|---:|---:|---:|
| | | |
| | | |
| | | |

What pattern do you notice?

**Your answer:**

_Write your answer here._


## 12. Plot the decision boundaries

The **decision boundary** is the line where a model changes from predicting class `0` to predicting class `1`.

The solid line is the boundary learned by our model. The dashed line is the boundary learned by scikit-learn.

The two lines should be close if our implementation is working correctly.


In [ ]:
from matplotlib.lines import Line2D

# Create a grid of possible exam-score pairs.
exam_1_grid = np.linspace(45, 100, 150)
exam_2_grid = np.linspace(45, 100, 150)
grid_x, grid_y = np.meshgrid(
    exam_1_grid,
    exam_2_grid
)

grid_points = np.column_stack(
    (grid_x.ravel(), grid_y.ravel())
)
grid_points_scaled = (
    grid_points - feature_mean
) / feature_std

our_grid_probability = model.predict_proba(
    grid_points_scaled
)[:, 1].reshape(grid_x.shape)

reference_grid_probability = reference_model.predict_proba(
    grid_points_scaled
)[:, 1].reshape(grid_x.shape)

plt.figure(figsize=(8, 6))
plt.scatter(
    X[not_admitted, 0],
    X[not_admitted, 1],
    label="Not admitted",
    marker="o"
)
plt.scatter(
    X[is_admitted, 0],
    X[is_admitted, 1],
    label="Admitted",
    marker="+"
)

plt.contour(
    grid_x,
    grid_y,
    our_grid_probability,
    levels=[0.5],
    linestyles="-"
)
plt.contour(
    grid_x,
    grid_y,
    reference_grid_probability,
    levels=[0.5],
    linestyles="--"
)

handles, labels = plt.gca().get_legend_handles_labels()
boundary_handles = [
    Line2D([0], [0], linestyle="-",
           label="Our model"),
    Line2D([0], [0], linestyle="--",
           label="scikit-learn")
]
plt.legend(handles=handles + boundary_handles)

plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.title("Decision Boundary Comparison")
plt.grid(True)
plt.show()


### Question 11

1. What does each decision boundary separate?
2. Are the two boundary lines close together?
3. Why might some points appear on the "wrong" side of both lines?

**Your answer:**

_Write your answer here._


## 13. Final reflection

Answer each question in one or two sentences.

1. Why is logistic regression appropriate for this problem?
2. How does gradient descent train the model?
3. What is the difference between a predicted probability and a predicted class?
4. Why did we compare our model with scikit-learn?
5. Does high accuracy prove that a model will always be correct? Explain.

**Your answer:**

_Write your answer here._


## Optional challenge

Change either the `learning_rate` or `number_of_steps`, then train the model again.

Compare:

- the final loss
- the test accuracy
- the agreement with scikit-learn

What happens when the model trains for too few steps?
